In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

df = pd.read_csv('history_traduzido.csv')

df['tempo_permanencia'] = df['tempo_permanencia'].fillna(0)
df['cobranca_total_acumulada'] = df['cobranca_total_acumulada'].fillna(0)

c = [
    (df['tempo_permanencia'] <= 3),
    (df['tempo_permanencia'] > 3) & (df['tempo_permanencia'] <= 12),
    (df['tempo_permanencia'] > 12) & (df['tempo_permanencia'] <= 24),
    (df['tempo_permanencia'] > 24)
]
v = ['0-3m', '4-12m', '1-2a', '+2a']
df['faixa_perm'] = np.select(c, v, default='desconhecido')

srv = ['seguranca_online', 'backup_online', 'protecao_dispositivos', 'suporte_tecnico', 'streaming_tv', 'streaming_filmes']
df['qtd_srv'] = df[srv].apply(lambda x: (x == 'Yes').sum(), axis=1)

df['cancelamento_servico'] = df['cancelamento_servico'].map({'Yes': 1, 'No': 0, 1: 1, 0: 0})
df['cancelamento_servico'] = df['cancelamento_servico'].fillna(0).astype(int)

def pega_fb(txt):
    if pd.isna(txt): return 'vazio'
    t = str(txt).lower()
    if any(k in t for k in ['slow', 'unreliable', 'disappointed']): return 'qualidade'
    elif any(k in t for k in ['check', 'mail in', 'payment']): return 'pagamento'
    elif any(k in t for k in ['steep', 'high', 'deal', 'pricing']): return 'preco'
    return 'outros'

df['cat_fb'] = df['feedback_cliente'].apply(pega_fb)

ccat = ['genero', 'possui_companheiro', 'possui_dependentes', 'servico_telefonico', 
        'multiplas_linhas_telefonicas', 'servico_internet', 'seguranca_online', 
        'backup_online', 'protecao_dispositivos', 'suporte_tecnico', 'streaming_tv', 
        'streaming_filmes', 'tipo_contrato', 'faturamento_digital', 'metodo_pagamento',
        'faixa_perm', 'cat_fb']

cnum = ['idoso', 'tempo_permanencia', 'cobranca_mensal', 'cobranca_total_acumulada', 'renda_mensal', 'qtd_srv']

enc = LabelEncoder()
for col in ccat:
    df[col] = enc.fit_transform(df[col].astype(str))

x = df[ccat + cnum]
y = df['cancelamento_servico']

xtr, xte, ytr, yte = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

p = (ytr == 0).sum() / (ytr == 1).sum()

mdl = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, scale_pos_weight=p, random_state=42, n_jobs=-1)
mdl.fit(xtr, ytr)

probs = mdl.predict_proba(xte)[:, 1]
preds = (probs >= 0.38).astype(int)

print(classification_report(yte, preds))

df['ltv_proj'] = df['cobranca_mensal'] * df['tempo_permanencia']


              precision    recall  f1-score   support

           0       0.92      0.79      0.85       826
           1       0.58      0.80      0.67       301

    accuracy                           0.79      1127
   macro avg       0.75      0.79      0.76      1127
weighted avg       0.83      0.79      0.80      1127

